# Manual Day — Python Essentials

Closed-agent, ~60 min. Complete every cell marked `# FILL IN`, then run the check cell.

In [1]:
import numpy as np

# Test matrices (GIVEN — do not change)
A = np.array([[2, 1],
              [1, 2]])   # known eigenvalues: 3 and 1
B = np.array([[4, 1],
              [2, 3]])   # known eigenvalues: 5 and 2
print(A)
print(B)

[[2 1]
 [1 2]]
[[4 1]
 [2 3]]


## 1. Eigenvalues & determinant of a 2x2 by hand

For a 2x2 matrix `M = [[a, b], [c, d]]`:

- determinant: `det = a*d - b*c`
- trace: `trace = a + d`
- eigenvalues via the quadratic formula: `lam = (trace ± sqrt(trace**2 - 4*det)) / 2`

Do this **without** `np.linalg.eig` or `np.linalg.det`.

**Known answers:** matrix `A` has eigenvalues `{3, 1}` (trace 4, det 3); matrix `B` has eigenvalues `{5, 2}` (trace 7, det 10).

In [2]:
# FILL IN: for a 2x2 matrix M=[[a,b],[c,d]], compute det=a*d-b*c, trace=a+d,
#          and the two eigenvalues via the quadratic formula
#          (no np.linalg.eig / no np.linalg.det).
def eig2x2(M):
    a, b = M[0, 0], M[0, 1]
    c, d = M[1, 0], M[1, 1]
    det = a*d - b*c
    trace = a + d
    lam_1 = (trace + np.sqrt(trace ** 2 - 4 * det)) / 2
    lam_2 = (trace - np.sqrt(trace ** 2 - 4 * det)) / 2
    return det, trace, lam_1, lam_2

In [3]:
# CHECK (GIVEN)
for name, M in [("A", A), ("B", B)]:
    det, trace, lam1, lam2 = eig2x2(M)
    mine = np.sort([lam1, lam2])
    ref = np.sort(np.linalg.eig(M)[0].real)
    print(f"--- Matrix {name} ---")
    print(f"det   = {det:.4f}   (np.linalg.det = {np.linalg.det(M):.4f})")
    print(f"trace = {trace:.4f}")
    print(f"eigenvalues (mine) = {mine}")
    print(f"eigenvalues (eig)  = {ref}")
    print(f"lam1+lam2 == trace : {np.isclose(lam1 + lam2, trace)}")
    print(f"lam1*lam2 == det   : {np.isclose(lam1 * lam2, det)}")
    print(f"matches np.linalg.eig: {np.allclose(mine, ref)}")
    print()

--- Matrix A ---
det   = 3.0000   (np.linalg.det = 3.0000)
trace = 4.0000
eigenvalues (mine) = [1. 3.]
eigenvalues (eig)  = [1. 3.]
lam1+lam2 == trace : True
lam1*lam2 == det   : True
matches np.linalg.eig: True

--- Matrix B ---
det   = 10.0000   (np.linalg.det = 10.0000)
trace = 7.0000
eigenvalues (mine) = [2. 5.]
eigenvalues (eig)  = [2. 5.]
lam1+lam2 == trace : True
lam1*lam2 == det   : True
matches np.linalg.eig: True



## 2. Matrix multiplication from scratch

Multiply two matrices with a plain triple loop — `C[i, j] = sum_k X[i, k] * Y[k, j]` — using **no** `@`, `np.matmul`, or `np.dot`.

**Known answer:** your result must equal `P @ Q`.

In [4]:
# GIVEN small integer matrices
rng = np.random.default_rng(0)
P = rng.integers(0, 5, size=(3, 3))
Q = rng.integers(0, 5, size=(3, 3))
print(P)
print(Q)

[[4 3 2]
 [1 1 0]
 [0 0 0]]
[[4 3 4]
 [2 3 4]
 [3 3 2]]


In [5]:
# FILL IN: multiply two matrices with a triple loop (no @, np.matmul, or np.dot).
def matmul(X, Y):
    C = np.zeros(shape=(X.shape[0],Y.shape[1]))
    for i in range(X.shape[0]):
        for j in range(Y.shape[1]):
            for k in range(X.shape[1]):
                C[i,j] += X[i,k] * Y[k,j]
    return C

In [6]:
# CHECK (GIVEN)
C = matmul(P, Q)
print("mine:\n", C)
print("P@Q:\n", P @ Q)
print("match:", np.allclose(C, P @ Q))

mine:
 [[28. 27. 32.]
 [ 6.  6.  8.]
 [ 0.  0.  0.]]
P@Q:
 [[28 27 32]
 [ 6  6  8]
 [ 0  0  0]]
match: True


## 3. Challenge — the dominant eigenvalue by power iteration

You now have two pieces: `eig2x2`, which gets the eigenvalues of a 2x2 *exactly*,
and `matmul`, which multiplies matrices. Put them together and find the **largest**
eigenvalue a third way — iteratively, never touching the quadratic formula.

Pick a starting vector `v` and repeat two steps: multiply by `M`, then rescale `v`
back to unit length. Every multiplication stretches the dominant eigendirection by
`lam1` and the other one by `lam2`, so after `k` steps the unwanted piece has shrunk
by `(lam2/lam1)**k` and `v` has swung around to point along the dominant
eigenvector. Read the eigenvalue off it with the **Rayleigh quotient**

    lam = (v . M v) / (v . v)

and stop once `lam` stops changing by more than `tol`.

One thing to look for in the output: for `A` the eigenvalue comes out far more
accurate than the eigenvector. When `M` is symmetric the Rayleigh quotient is
*quadratically* accurate in the error of `v`, so `lam` settles to `tol` while `v` is
still only good to about `sqrt(tol)`. `B` is not symmetric, gets no such bonus, and
its eigenvalue and eigenvector arrive together. A good *number* from a mediocre
*vector* is a recurring gift in numerical physics — and a trap if you assume the
reverse.

**Known answers:** `A` -> 3 and `B` -> 5, the larger eigenvalues you already
computed in Problem 1. Convergence is set by `|lam2/lam1|` — 1/3 for `A`, 2/5 for
`B` — so a few dozen steps is plenty. The check also verifies `M v = lam v`, i.e.
that you found the eigen*vector* too.

Why this one matters beyond linear algebra: it is the first **iterative** algorithm
of the course. A loop that runs until a number stops moving is exactly the shape of
gradient descent next week, of Euler and Jacobi relaxation in week 4, and of
Metropolis later on. It is also how you get the dominant eigenvalue of a transfer
matrix, or the top principal component, when the matrix is far too big to
diagonalize.

In [7]:
# GIVEN — starting vector and stopping controls.
# v0 = (1, 0) on purpose: for A the vector (1, 1) is *already* the dominant
# eigenvector, so starting there would converge in one step and teach nothing.
v0 = np.array([[1.0], [0.0]])
tol = 1e-12
max_iter = 1000

In [8]:
# FILL IN: power iteration — multiply by M with your own matmul, renormalize,
#          read lam off the Rayleigh quotient, stop when lam settles.
def power_iteration(M, v0, tol, max_iter):
    v = v0 / np.sqrt(np.sum(v0**2))                      # start from a unit vector
    lam_old = 0.0
    for it in range(max_iter):
        w = matmul(M,v)                                                  # multiply by M (reuse matmul)
        v = w / np.sqrt(np.sum(v**2))                                    # rescale to unit length
        lam = (matmul(v,w))/ (np.sum(v**2))                              # Rayleigh quotient (v.Mv)/(v.v)
        if abs(lam - lam_old) < tol:                                     # lam stopped moving by more than tol
            break
        lam_old = lam
    return lam, v[:, 0], it + 1

In [9]:
# CHECK (GIVEN)
for name, M in [("A", A), ("B", B)]:
    lam, v, iters = power_iteration(M, v0, tol, max_iter)
    lam_exact = max(eig2x2(M)[2:])                       # larger eigenvalue, Problem 1
    resid = np.max(np.abs(matmul(M, v.reshape(-1, 1))[:, 0] - lam * v))
    print(f"--- Matrix {name} ---")
    print(f"power iteration lam = {lam:.12f}  in {iters} steps")
    print(f"exact (Problem 1)   = {lam_exact:.12f}")
    print(f"eigenvector v       = {v}")
    print(f"|M v - lam v|       = {resid:.2e}")
    assert np.isclose(lam, lam_exact), "did not converge to the dominant eigenvalue"
    assert resid < 1e-4, "v is not an eigenvector of M"
    print()
print("PASSED: dominant eigenvalue and eigenvector recovered by iteration")
print("Compare the two residuals: A is symmetric, so lam converges quadratically")
print("and stops while v is still rough. B gets no such bonus and arrives together.")

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()